In [11]:
# --- Import libraries ---
import pandas as pd
import statsmodels.formula.api as smf
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import os 

In [259]:
# --- Import data ---
exp1_path =  "/Users/elisavanderplas/Desktop/vdPlas_2025_Politics/exp1/behavioural_exp1.csv"
exp2_path =  "/Users/elisavanderplas/Desktop/vdPlas_2025_Politics/exp2/behavioural_exp2.csv"
exp2_demo_path =  "/Users/elisavanderplas/Desktop/vdPlas_2025_Politics/exp2/subjectLog_exp2.csv"
data_exp1 = pd.read_csv(exp1_path, sep = ';')
data_exp2 = pd.read_csv(exp2_path, sep = ',')
demo_exp2 = pd.read_csv(exp2_demo_path) 

In [ ]:
# --- Prep data Exp 1 --- 
data_exp1['Video'] = pd.Categorical(data_exp1['Video'], categories = [1,2,3,4,5,6,7,8,9], ordered=False)
data_exp1['Video'] = data_exp1['Video'].cat.rename_categories(["immigration_threat","immigration_blame","immigration_uncertain", "environment_threat","environment_blame","environment_uncertain", "healthcare_threat","healthcare_blame","healthcare_uncertain"])

data_exp1['topic'] = pd.Categorical(data_exp1['video_2'], categories = [1,2,3], ordered=False)
data_exp1['topic'] = data_exp1['topic'].cat.rename_categories(["immigration","environment","healthcare"])

data_exp1['frame'] = pd.Categorical(data_exp1['video_3'], categories = [1,2,3], ordered=False)
data_exp1['frame'] = data_exp1['frame'].cat.rename_categories(["threat","blame","uncertain"])

data_exp1['gender'] = data_exp1['geslacht'] - 1 #back to 0 and 1 with 0 is male
data_exp1['online_use'] = data_exp1['V1_6'] #1 = Uses Social Media (such as Facebook, Twitter or Youtube) to follow the news
data_exp1['paper_read'] = data_exp1['V2a'] #1 = Reads political news in newspapers
data_exp1['online_read'] = data_exp1['V2b'] #1 = Reads political news o social media
data_exp1['interest_politics'] = data_exp1['V3'] #1-4 Likert from "Very interested in political issues" to "Not interested at all"
data_exp1['CDA'] = data_exp1['V5_1']
data_exp1['D66'] = data_exp1['V5_2']
data_exp1['GL'] = data_exp1['V5_3']
data_exp1['PvdA'] = data_exp1['V5_4']
data_exp1['PVV'] = data_exp1['V5_5']
data_exp1['SP'] = data_exp1['V5_6']
data_exp1['VVD'] = data_exp1['V5_7']
data_exp1['50P'] = data_exp1['V5_8']
data_exp1['leftright'] = data_exp1['V_6']

data_exp1['importance_gezondheidsz'] = data_exp1['V13_1']
data_exp1['importance_immigr'] = data_exp1['V13_2']
data_exp1['importance_milieu'] = data_exp1['V13_3']

data_exp1['video_agreement'] = data_exp1['V10_4']
data_exp1['video_realiability'] =data_exp1['V10_5']
data_exp1['video_sharing'] = data_exp1['V10_6']

# Drop columns that start with 'V'
cols_to_drop = [col for col in data_exp1.columns if col.startswith('V')]
df_exp1 = data_exp1.drop(columns=cols_to_drop)

# Identify columns that start with 'INDEX'
index_cols = [col for col in df_exp1.columns if col.startswith(('INDEX', 'BIG', 'online', 'geboortejaar', 'paper'))]

# Clean and replace
for col in index_cols:
    # Convert 'NULL#' to NaN
    df_exp1[col] = df_exp1[col].replace('NULL#', np.nan)
    
    # Convert column to numeric (in case it's still object type)
    df_exp1[col] = pd.to_numeric(df_exp1[col], errors='coerce')
    
    # Compute median (ignoring NaN)
    median_value = df_exp1[col].median()
    
    # Fill NaN with median and convert to int
    df_exp1[col] = df_exp1[col].fillna(median_value).astype(float)

# Set 'uncertain' as reference category using pandas Categorical
df_exp1['frame_reference'] = pd.Categorical(df_exp1['frame'], categories=['uncertain', 'threat', 'blame'])


Study 1 

In [246]:
report_text = f"In Study 1 we analyzed the data of {len(df_exp1)} Dutch adults ({np.mean(df_exp1.gender)*100}% female)"

print(report_text)

In Study 1 we analyzed the data of 1825 Dutch adults (53.36986301369863% female)


In [173]:
# --- Model 1 : Negative Arousal ---
model_neg_arousal = smf.ols(
    'NEG_AFFECT ~ C(frame_reference) + BIG5_O + BIG5_C + BIG5_E + BIG5_A + BIG5_N + online_read + interest_politics + opleiding + lft4 + gender + leftright + INDEX_AUTHO + INDEX_MIGRATION + INDEX_CLIMATE + INDEX_HEALTH + INDEX_CYNICISM ',
    data=df_exp1
).fit()
print(model_neg_arousal.summary())


model_pos_arousal = smf.ols(
    'POS_AFFECT ~ C(frame_reference) + BIG5_O + BIG5_C + BIG5_E + BIG5_A + BIG5_N + online_read + interest_politics + opleiding + lft4 + gender + leftright + INDEX_AUTHO + INDEX_MIGRATION + INDEX_CLIMATE + INDEX_HEALTH + INDEX_CYNICISM ',
    data=df_exp1
).fit()
print(model_pos_arousal.summary())


                            OLS Regression Results                            
Dep. Variable:             NEG_AFFECT   R-squared:                       0.072
Model:                            OLS   Adj. R-squared:                  0.063
Method:                 Least Squares   F-statistic:                     7.837
Date:                Sun, 19 Oct 2025   Prob (F-statistic):           3.13e-20
Time:                        16:24:11   Log-Likelihood:                -4014.7
No. Observations:                1825   AIC:                             8067.
Df Residuals:                    1806   BIC:                             8172.
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

In [174]:
# --- Model 2 : Fear---
model_fear = smf.ols(
    'FEAR ~ C(frame_reference) +  BIG5_O + BIG5_C + BIG5_E + BIG5_A + BIG5_N + online_read + interest_politics + opleiding + lft4 + gender + leftright + INDEX_AUTHO + INDEX_MIGRATION + INDEX_CLIMATE + INDEX_HEALTH + INDEX_CYNICISM ',
    data=df_exp1
).fit()
print(model_fear.summary())

                            OLS Regression Results                            
Dep. Variable:                   FEAR   R-squared:                       0.092
Model:                            OLS   Adj. R-squared:                  0.083
Method:                 Least Squares   F-statistic:                     10.16
Date:                Sun, 19 Oct 2025   Prob (F-statistic):           9.27e-28
Time:                        16:24:13   Log-Likelihood:                -5478.8
No. Observations:                1825   AIC:                         1.100e+04
Df Residuals:                    1806   BIC:                         1.110e+04
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

In [300]:
# --- Model 3 : Anger ---
model_anger = smf.ols(
    'ANGER ~ C(frame_reference) +  BIG5_O + BIG5_C + BIG5_E + BIG5_A + BIG5_N + online_read + interest_politics  + opleiding + lft4 + gender + leftright + INDEX_AUTHO + INDEX_MIGRATION + INDEX_CLIMATE + INDEX_HEALTH + INDEX_CYNICISM  ',
    data=df_exp1
).fit()
print(model_anger.summary())

                            OLS Regression Results                            
Dep. Variable:                  ANGER   R-squared:                       0.066
Model:                            OLS   Adj. R-squared:                  0.057
Method:                 Least Squares   F-statistic:                     7.088
Date:                Sun, 19 Oct 2025   Prob (F-statistic):           7.97e-18
Time:                        21:30:53   Log-Likelihood:                -5509.7
No. Observations:                1825   AIC:                         1.106e+04
Df Residuals:                    1806   BIC:                         1.116e+04
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

In [315]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# Extract coefficients from Study 1 anger model
def extract_study1_coefficients(model):
    """Extract threat and blame coefficients from Study 1 model"""
    coefficients = {}
    
    for param in model.params.index:
        if 'frame_reference' in param:
            if '[T.threat]' in param:
                coefficients['threat'] = {
                    'coef': model.params[param],
                    'se': model.bse[param],
                    'p_value': model.pvalues[param]
                }
                print(f"Study 1 Anger - Threat: {model.params[param]:.4f} (SE: {model.bse[param]:.4f}, p: {model.pvalues[param]:.4f})")
            elif '[T.blame]' in param:
                coefficients['blame'] = {
                    'coef': model.params[param],
                    'se': model.bse[param],
                    'p_value': model.pvalues[param]
                }
                print(f"Study 1 Anger - Blame: {model.params[param]:.4f} (SE: {model.bse[param]:.4f}, p: {model.pvalues[param]:.4f})")
    
    return coefficients

# Extract coefficients from your Study 1 models
study1_anger_coefs = extract_study1_coefficients(model_anger)
study1_fear_coefs = {
    'threat': {'coef': 8.2201, 'se': 2.543, 'p_value': 0.001},
    'blame': {'coef': 4.9440, 'se': 2.395, 'p_value': 0.039}
}

# Create the updated plot with Study 1 coefficients
fig = go.Figure()

# Define conditions and colors
conditions = ['Threat', 'Blame']
fear_color = '#56B4E9'   # light blue
anger_color = '#D55E00'  # dark pink / orange-red

# Add Fear data 
fig.add_trace(go.Scatter(
    name='Fear',
    x=conditions,
    y=[study1_fear_coefs['threat']['coef'], study1_fear_coefs['blame']['coef']],
    error_y=dict(
        type='data',
        array=[study1_fear_coefs['threat']['se'], study1_fear_coefs['blame']['se']],
        visible=True,
        thickness=3,
        width=6,
        color='black'  
    ),
    mode='markers', 
    marker=dict(
        size=16,
        color=fear_color,
        symbol='diamond',
        line=dict(width=3, color='black')
    ),
    hovertemplate='Fear - %{x}: %{y:.2f} ± %{error_y.array:.2f}<extra></extra>'
))

# Add Anger data 
fig.add_trace(go.Scatter(
    name='Anger',
    x=conditions,
    y=[study1_anger_coefs['threat']['coef'], study1_anger_coefs['blame']['coef']],
    error_y=dict(
        type='data',
        array=[study1_anger_coefs['threat']['se'], study1_anger_coefs['blame']['se']],
        visible=True,
        thickness=3,
        width=6,
        color='black'  #
    ),
    mode='markers',
    marker=dict(
        size=16,
        color=anger_color,
        symbol='diamond',
        line=dict(width=3, color='black')
    ),
    hovertemplate='Anger - %{x}: %{y:.2f} ± %{error_y.array:.2f}<extra></extra>'
))

# Calculate significance stars based on p-values
def get_significance_stars(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return 'n.s.'

# Add significance annotations
y_offset_fear = max(study1_fear_coefs['threat']['coef'] + study1_fear_coefs['threat']['se'], 
                   study1_fear_coefs['blame']['coef'] + study1_fear_coefs['blame']['se']) * 0.2

y_offset_anger = max(study1_anger_coefs['threat']['coef'] + study1_anger_coefs['threat']['se'], 
                    study1_anger_coefs['blame']['coef'] + study1_anger_coefs['blame']['se']) * 0.2

# Fear significance annotations
fig.add_annotation(
    x='Threat', 
    y=study1_fear_coefs['threat']['coef'] + study1_fear_coefs['threat']['se'] + y_offset_fear,
    text=get_significance_stars(study1_fear_coefs['threat']['p_value']),
    showarrow=False, 
    font=dict(
        size=22 if get_significance_stars(study1_fear_coefs['threat']['p_value']) != 'n.s.' else 16,
        color='black', 
        family='Arial Black'
    ),
    yshift=10
)

fig.add_annotation(
    x='Blame', 
    y=study1_fear_coefs['blame']['coef'] + study1_fear_coefs['blame']['se'] + y_offset_fear,
    text=get_significance_stars(study1_fear_coefs['blame']['p_value']),
    showarrow=False, 
    font=dict(
        size=22 if get_significance_stars(study1_fear_coefs['blame']['p_value']) != 'n.s.' else 16,
        color='black', 
        family='Arial Black'
    ),
    yshift=10
)

# Anger significance annotations
fig.add_annotation(
    x='Threat', 
    y=study1_anger_coefs['threat']['coef'] + study1_anger_coefs['threat']['se'] + y_offset_anger,
    text=get_significance_stars(study1_anger_coefs['threat']['p_value']),
    showarrow=False, 
    font=dict(
        size=22 if get_significance_stars(study1_anger_coefs['threat']['p_value']) != 'n.s.' else 16,
        color='black', 
        family='Arial Black'
    ),
    yshift=10
)

fig.add_annotation(
    x='Blame', 
    y=study1_anger_coefs['blame']['coef'] + study1_anger_coefs['blame']['se'] + y_offset_anger,
    text=get_significance_stars(study1_anger_coefs['blame']['p_value']),
    showarrow=False, 
    font=dict(
        size=22 if get_significance_stars(study1_anger_coefs['blame']['p_value']) != 'n.s.' else 16,
        color='black', 
        family='Arial Black'
    ),
    yshift=10
)

# Customize layout
fig.update_layout(
    title=dict(
        text="<b>(b.) Study 1</b>",
        x=0.05, 
        xanchor='left',  
        font=dict(size=20, family='Arial Black')
    ),
    xaxis=dict(
        title="",
        tickmode='array',
        tickvals=conditions,
        ticktext=conditions,
        tickfont=dict(size=16, family='Arial'),
        showgrid=False,
        range=[-0.3, 1.3] 
    ),
    yaxis=dict(
        title="Simplification frame impact<br>on negative arousal (a.u.)",  
        title_font=dict(size=18, family='Arial'),  
        tickfont=dict(size=14, family='Arial'),
        gridcolor='lightgray',
        gridwidth=1,
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor='black'
    ),
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(size=14, family='Arial'),
        bgcolor='rgba(255,255,255,0.8)'
    ),
    template="plotly_white",
    width=400,
    height=400,
    margin=dict(l=80, r=40, t=80, b=60)
)

# Add a subtle background
fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig.show()


Study 1 Anger - Threat: 1.2582 (SE: 0.2878, p: 0.0000)
Study 1 Anger - Blame: 2.0601 (SE: 0.3350, p: 0.0000)


In [179]:

# --- Model 4A : PVV attitudes ---
model_pvv = smf.ols(
    'PVV ~ C(topic) + NEG_AFFECT + C(frame) + C(topic):NEG_AFFECT + BIG5_O + BIG5_C + BIG5_E + BIG5_A + BIG5_N + online_read + interest_politics  + opleiding + lft4 + gender + leftright + INDEX_AUTHO + INDEX_MIGRATION + INDEX_CLIMATE + INDEX_HEALTH + INDEX_CYNICISM + INDEX_MIGRATION :C(frame_reference) +  INDEX_MIGRATION :C(frame_reference):NEG_AFFECT',
    data=df_exp1
).fit()
print(model_pvv.summary())



# --- Model 4B : GL attitudes ---
model_gl = smf.ols(
    'GL ~ C(topic) + NEG_AFFECT + C(frame)+ C(topic):NEG_AFFECT  + BIG5_O + BIG5_C + BIG5_E + BIG5_A + BIG5_N + online_read + interest_politics  + opleiding + lft4 + gender + leftright + INDEX_AUTHO + INDEX_MIGRATION + INDEX_CLIMATE + INDEX_HEALTH + INDEX_CYNICISM + INDEX_CLIMATE:C(frame_reference) + INDEX_CLIMATE:C(frame_reference):NEG_AFFECT',
    data=df_exp1
).fit()
print(model_gl.summary())


                            OLS Regression Results                            
Dep. Variable:                    PVV   R-squared:                       0.375
Model:                            OLS   Adj. R-squared:                  0.365
Method:                 Least Squares   F-statistic:                     38.41
Date:                Sun, 19 Oct 2025   Prob (F-statistic):          1.23e-160
Time:                        16:25:48   Log-Likelihood:                -5429.5
No. Observations:                1825   AIC:                         1.092e+04
Df Residuals:                    1796   BIC:                         1.108e+04
Df Model:                          28                                         
Covariance Type:            nonrobust                                         
                                                               coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Create subplot figure with 1 row and 2 columns
fig = make_subplots(rows=1, cols=2, 
                   subplot_titles=('A) GL Attitudes by Negative Affect and Topic', 
                                  'B) PVV Attitudes by Negative Affect and Topic'),
                   shared_yaxes=False)

# Define topics and colors
topics = ['immigration', 'environment', 'healthcare']
colors = ['red', 'green', 'blue']  # red for immigration, green for environment, blue for healthcare

# negative affect values for prediction
neg_affect_range = np.linspace(df_exp1['NEG_AFFECT'].min(), df_exp1['NEG_AFFECT'].max(), 50)

# --- LEFT PLOT: GL Attitudes by Negative Affect ---
for i, topic in enumerate(topics):
    # Calculate predicted GL attitudes for this topic across negative affect range
    if topic == 'immigration':
        # Reference category for topic
        topic_effect = 0
        topic_affect_interaction = 0
    elif topic == 'environment':
        topic_effect = -0.6007  # C(topic)[T.environment]
        topic_affect_interaction = 0.2263  # C(topic)[T.environment]:NEG_AFFECT
    else:  # healthcare
        topic_effect = -0.3747  # C(topic)[T.healthcare]
        topic_affect_interaction = 0.1127  # C(topic)[T.healthcare]:NEG_AFFECT
    
    predicted_gl = (11.1089 +  # Intercept
                   topic_effect + 
                   neg_affect_range * 0.0335 +  # NEG_AFFECT main effect
                   neg_affect_range * topic_affect_interaction)
    
    fig.add_trace(
        go.Scatter(x=neg_affect_range, y=predicted_gl,
                  mode='lines',
                  name=f'{topic}',
                  line=dict(color=colors[i], width=3),
                  showlegend=True),
        row=1, col=1
    )

# --- RIGHT PLOT: PVV Attitudes by Negative Affect ---
for i, topic in enumerate(topics):
    # Calculate predicted PVV attitudes for this topic across negative affect range
    if topic == 'immigration':
        # Reference category for topic
        topic_effect = 0
        topic_affect_interaction = 0
    elif topic == 'environment':
        topic_effect = 1.4084  # C(topic)[T.environment]
        topic_affect_interaction = -0.2823  # C(topic)[T.environment]:NEG_AFFECT
    else:  # healthcare
        topic_effect = 0.9733  # C(topic)[T.healthcare]
        topic_affect_interaction = -0.3822  # C(topic)[T.healthcare]:NEG_AFFECT
    
    predicted_pvv = (5.2341 +  # Intercept
                    topic_effect + 
                    neg_affect_range * 0.3433 +  # NEG_AFFECT main effect
                    neg_affect_range * topic_affect_interaction)
    
    fig.add_trace(
        go.Scatter(x=neg_affect_range, y=predicted_pvv,
                  mode='lines',
                  name=f'PVV - {topic}',
                  line=dict(color=colors[i], width=3, dash='dash'),
                  showlegend=False),
        row=1, col=2
    )

# Update layout
fig.update_layout(
    title_text="Party Attitudes by Negative Affect and Video Topic",
    height=500,
    width=1000,
    template="plotly_white"
)

# Update axes labels
fig.update_xaxes(title_text="Negative Affect", row=1, col=1)
fig.update_xaxes(title_text="Negative Affect", row=1, col=2)
fig.update_yaxes(title_text="GL Attitudes", row=1, col=1)
fig.update_yaxes(title_text="PVV Attitudes", row=1, col=2)

fig.show()

In [283]:
# --- Model 4A :Agreement ---
model_agreement = smf.ols(
    'video_agreement ~ C(frame):NEG_AFFECT + NEG_AFFECT + C(frame) + C(topic) +  BIG5_O + BIG5_C + BIG5_E + BIG5_A + BIG5_N + opleiding + lft4 + gender + leftright + INDEX_AUTHO + INDEX_MIGRATION + INDEX_CLIMATE + INDEX_HEALTH + INDEX_CYNICISM ',
    data=df_exp1
).fit()
print(model_agreement.summary())

                            OLS Regression Results                            
Dep. Variable:        video_agreement   R-squared:                       0.316
Model:                            OLS   Adj. R-squared:                  0.308
Method:                 Least Squares   F-statistic:                     39.60
Date:                Sun, 19 Oct 2025   Prob (F-statistic):          6.31e-132
Time:                        20:59:47   Log-Likelihood:                -2621.8
No. Observations:                1825   AIC:                             5288.
Df Residuals:                    1803   BIC:                             5409.
Df Model:                          21                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercep

In [196]:
# Create overall importance as mean of the three indices
df_exp1['ISSUE_IMPORTANCE'] = df_exp1[['INDEX_CLIMATE', 'INDEX_HEALTH', 'INDEX_CYNICISM']].mean(axis=1)

# Run the model with overall importance
model_importance= smf.ols(
    'ISSUE_IMPORTANCE ~ NEG_AFFECT + C(frame_reference) + NEG_AFFECT + C(frame_reference) + BIG5_O + BIG5_C + BIG5_E + BIG5_A + BIG5_N + opleiding + lft4 + gender + leftright + INDEX_AUTHO + INDEX_CYNICISM',
    data=df_exp1
).fit()

print(model_importance.summary())

                            OLS Regression Results                            
Dep. Variable:       ISSUE_IMPORTANCE   R-squared:                       0.752
Model:                            OLS   Adj. R-squared:                  0.750
Method:                 Least Squares   F-statistic:                     392.8
Date:                Sun, 19 Oct 2025   Prob (F-statistic):               0.00
Time:                        16:54:46   Log-Likelihood:                -2400.5
No. Observations:                1825   AIC:                             4831.
Df Residuals:                    1810   BIC:                             4914.
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

In [277]:
## Model 5 - Sharing
model_sharing = smf.ols(
    'video_sharing ~ NEG_AFFECT + C(frame_reference) + NEG_AFFECT + BIG5_O + BIG5_C + BIG5_E + BIG5_A + BIG5_N + opleiding + lft4 + gender + leftright + INDEX_AUTHO + INDEX_MIGRATION  + INDEX_CLIMATE + INDEX_HEALTH + INDEX_CYNICISM ',
    data=df_exp1
).fit()
print(model_sharing.summary())

                            OLS Regression Results                            
Dep. Variable:          video_sharing   R-squared:                       0.243
Model:                            OLS   Adj. R-squared:                  0.236
Method:                 Least Squares   F-statistic:                     34.07
Date:                Sun, 19 Oct 2025   Prob (F-statistic):           2.23e-96
Time:                        20:46:28   Log-Likelihood:                -2974.5
No. Observations:                1825   AIC:                             5985.
Df Residuals:                    1807   BIC:                             6084.
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# Create the plot with the specified color scheme and formatting
fig = go.Figure()

# Define conditions and colors from your specification
conditions = ['Threat', 'Blame']
sharing_color = '#2C4A52'    # Dark teal (agreement/upper diamond)
importance_color = '#E6C87F' # Muted yellow-beige (importance/lower diamond)
error_color = '#222222'      # Black (slightly gray)
grid_color = '#D3D3D3'       # Light gray

# Adjust coefficients by moving everything 0.5 down on y-axis
adjusted_sharing_threat = sharing_coefs['threat']['coef'] - 0.5
adjusted_sharing_blame = sharing_coefs['blame']['coef'] - 0.5
adjusted_importance_threat = importance_coefs['threat']['coef'] - 0.5
adjusted_importance_blame = importance_coefs['blame']['coef'] - 0.5

# Add Video Sharing data with diamond markers (NO connecting lines)
fig.add_trace(go.Scatter(
    name='Video Sharing',
    x=conditions,
    y=[adjusted_sharing_threat, adjusted_sharing_blame],
    error_y=dict(
        type='data',
        array=[sharing_coefs['threat']['se'], sharing_coefs['blame']['se']],
        visible=True,
        thickness=3,
        width=6,
        color=error_color  
    ),
    mode='markers', 
    marker=dict(
        size=20,  
        color=sharing_color,
        symbol='diamond',
        line=dict(width=3, color=error_color)
    ),
    hovertemplate='Video Sharing - %{x}: %{y:.2f} ± %{error_y.array:.2f}<extra></extra>'
))

# Add Issue Importance data with diamond markers
fig.add_trace(go.Scatter(
    name='Issue Importance',
    x=conditions,
    y=[adjusted_importance_threat, adjusted_importance_blame],
    error_y=dict(
        type='data',
        array=[importance_coefs['threat']['se'], importance_coefs['blame']['se']],
        visible=True,
        thickness=3,
        width=6,
        color=error_color 
    ),
    mode='markers',  
    marker=dict(
        size=20, 
        color=importance_color,
        symbol='diamond',
        line=dict(width=3, color=error_color)
    ),
    hovertemplate='Issue Importance - %{x}: %{y:.2f} ± %{error_y.array:.2f}<extra></extra>'
))

# Calculate significance stars based on p-values
def get_significance_stars(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return 'n.s.'

# Add significance annotations 
y_offset = 0.3 

# Video Sharing significance annotations
fig.add_annotation(
    x='Threat', 
    y=adjusted_sharing_threat + sharing_coefs['threat']['se'] + y_offset,
    text=get_significance_stars(sharing_coefs['threat']['p_value']),
    showarrow=False, 
    font=dict(
        size=20 if get_significance_stars(sharing_coefs['threat']['p_value']) != 'n.s.' else 14,  # LESS BOLD sizing
        color='black', 
        family='Arial' 
    ),
    yshift=10
)

fig.add_annotation(
    x='Blame', 
    y=adjusted_sharing_blame + sharing_coefs['blame']['se'] + y_offset,
    text=get_significance_stars(sharing_coefs['blame']['p_value']),
    showarrow=False, 
    font=dict(
        size=20 if get_significance_stars(sharing_coefs['blame']['p_value']) != 'n.s.' else 14,  # LESS BOLD sizing
        color='black', 
        family='Arial'  
    ),
    yshift=10
)

# Issue Importance significance annotations
fig.add_annotation(
    x='Threat', 
    y=adjusted_importance_threat + importance_coefs['threat']['se'] + y_offset,
    text=get_significance_stars(importance_coefs['threat']['p_value']),
    showarrow=False, 
    font=dict(
        size=20 if get_significance_stars(importance_coefs['threat']['p_value']) != 'n.s.' else 14, 
        color='black', 
        family='Arial'  
    ),
    yshift=10
)

fig.add_annotation(
    x='Blame', 
    y=adjusted_importance_blame + importance_coefs['blame']['se'] + y_offset,
    text=get_significance_stars(importance_coefs['blame']['p_value']),
    showarrow=False, 
    font=dict(
        size=20 if get_significance_stars(importance_coefs['blame']['p_value']) != 'n.s.' else 14,  
        color='black', 
        family='Arial'  
    ),
    yshift=10
)

# Customize layout
fig.update_layout(
    title=dict(
        text="(a.) Study 1", 
        x=0.05,  
        xanchor='left',  
        font=dict(size=22, family='Arial') 
    ),
    xaxis=dict(
        title="",
        tickmode='array',
        tickvals=conditions,
        ticktext=conditions,
        tickfont=dict(size=18, family='Arial'), 
        showgrid=False,
        range=[-0.3, 1.3]  
    ),
    yaxis=dict(
        title="Simplification frame impact<br>on political behaviour (a.u.)",  
        title_font=dict(size=20, family='Arial'), 
        tickfont=dict(size=16, family='Arial'), 
        gridcolor=grid_color,
        gridwidth=1,
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor='black'
    ),
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(size=16, family='Arial'), 
        bgcolor='rgba(255,255,255,0.8)'
    ),
    template="plotly_white",
    width=400,
    height=500,
    margin=dict(l=80, r=40, t=80, b=60)
)

fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig.show()



STUDY 1 COEFFICIENTS FOR POLITICAL BEHAVIOR
ORIGINAL COEFFICIENTS (before -0.5 adjustment):
VIDEO SHARING:
  Threat: β = -0.060, SE = 0.082, p = 0.4660
  Blame:  β = -0.139, SE = 0.095, p = 0.1429

ISSUE IMPORTANCE:
  Threat: β = 0.084, SE = 0.104, p = 0.4199
  Blame:  β = 0.228, SE = 0.121, p = 0.0604

Note: All y-values have been adjusted by -0.5 for visualization


Study 2

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.anova import anova_lm
from sklearn.preprocessing import StandardScaler

# Factors - Convert to categorical with labels
data_exp2['subj_idx'] = data_exp2['subj_idx'].astype('category')

# Create factors
data_exp2['Topic'] = data_exp2['Topic'].replace({1: 'immigration', 2: 'climate change', 3: 'healthcare'}).astype('category')
data_exp2['Appraisal'] = data_exp2['Appraisal '].replace({1: 'uncertain', 2: 'threat', 3: 'blame'}).astype('category')

# Merge demographics from demo_exp2
expected_demo_cols = ['sj_nr', 'gender2', 'age', 'education', 'pol_autho', 'pol_scep', 
                     'issue_positions_immigration', 'issue_positions_climate', 'issue_positions_health']

# Find which expected columns actually exist
available_demo_cols = [col for col in expected_demo_cols if col in demo_exp2.columns]
print(f"Available demographic columns for merging: {available_demo_cols}")

# Merge demographics 
data_exp2 = data_exp2.merge(
    demo_exp2[available_demo_cols],
    left_on='subj_idx',
    right_on='sj_nr',
    how='left'
)

print(f"Study 2 data shape after merge: {data_exp2.shape}")

# Rename columns 
rename_mapping = {}
if 'gender2' in data_exp2.columns:
    rename_mapping['gender2'] = 'gender'
if 'education' in data_exp2.columns:
    rename_mapping['education'] = 'edu'
if 'issue_positions_immigration' in data_exp2.columns:
    rename_mapping['issue_positions_immigration'] = 'PositionImm'
if 'issue_positions_climate' in data_exp2.columns:
    rename_mapping['issue_positions_climate'] = 'PositionCl'
if 'issue_positions_health' in data_exp2.columns:
    rename_mapping['issue_positions_health'] = 'PositionHe'

data_exp2 = data_exp2.rename(columns=rename_mapping)

#  Set 'uncertain' as reference category using pandas Categorical
data_exp2['frame_reference'] = pd.Categorical(data_exp2['Appraisal'], categories=['uncertain', 'threat', 'blame'])
# For clarity
data_exp2['video_sharing'] = data_exp2['clap']


Topic categories: ['climate change', 'healthcare', 'immigration']
Appraisal categories: ['blame', 'threat', 'uncertain']

Available demographics columns: ['sj_nr', 'sj_nr_Skyra', 'date', 'time', 'SONA status', 'lab', 'sj_nr_SONA2', 'gender', 'gender2', 'age', 'education', 'employment', 'big5_E', 'big5_N', 'big5_O', 'big5_C', 'big5_A', 'issue_positions_immigration', 'issue_positions_climate', 'issue_positions_health', 'issue_importance_immigration', 'issue_importance_climate', 'issue_importance_health', 'party_attitude_PVV', 'party_attitude_GL', 'party_attitude_SP', 'victim_immigration', 'victim_climate', 'victim_health', 'control_health', 'control_climatek', 'control_immigration', 'pol_scep', 'pol_autho']
Available demographic columns for merging: ['sj_nr', 'gender2', 'age', 'education', 'pol_autho', 'pol_scep', 'issue_positions_immigration', 'issue_positions_climate', 'issue_positions_health']
Study 2 data shape after merge: (1077, 39)


In [281]:
report_text = f"In Study 2 we analyzed the data of {len(data_exp2.sj_nr.unique())} Dutch adults ({np.mean(data_exp2.gender)*100}% female))"

print(report_text)

In Study 2 we analyzed the data of 27 Dutch adults (50.836120401337794% female))


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols
import matplotlib.pyplot as plt
import seaborn as sns

# Negative Affect model
negative_appraisal = ols('NEG_AFFECT ~ C(frame_reference) + age + gender + edu + pol_autho + pol_scep + PositionImm + PositionCl + PositionHe', data=data_exp2).fit()
print("=== NEGATIVE AFFECT MODEL ===")
print(negative_appraisal.summary())
print("\n")

# Fear model
fear_appraisal = ols('FEAR ~ C(frame_reference) + age + gender + edu + pol_autho + pol_scep + PositionImm + PositionCl + PositionHe', data=data_exp2).fit()
print("=== FEAR MODEL ===")
print(fear_appraisal.summary())
print("\n")

# Anger model  
anger_appraisal = ols('ANGER ~ C(frame_reference) + age + gender + edu + pol_autho + pol_scep + PositionImm + PositionCl + PositionHe', data=data_exp2).fit()
print("=== ANGER MODEL ===")
print(anger_appraisal.summary())
print("\n")

# PART 2: BEHAVIOURAL EFFECTS

# Importance model
imp_appraisal = ols('importance ~ C(frame_reference) + NEG_AFFECT + age + gender + edu + pol_autho + pol_scep + PositionImm + PositionCl + PositionHe', data=data_exp2).fit()
print("=== IMPORTANCE MODEL ===")
print(imp_appraisal.summary())
print("\n")

# Clap model
clap_appraisal = ols('video_sharing ~ C(frame_reference) + NEG_AFFECT + age + gender + edu + pol_autho + pol_scep + PositionImm + PositionCl + PositionHe', data=data_exp2).fit()
print("=== CLAP MODEL ===")
print(clap_appraisal.summary())
print("\n")


Column names in data_exp2:
['subj_idx', 'Topic', 'Appraisal ', 'clap', 'donatie', 'importance', 'FEAR', 'ANGER', 'NEG_AFFECT', 'Appraisal', 'sj_nr_x', 'gender', 'age_x', 'edu', 'pol_autho_x', 'pol_scep_x', 'PositionImm', 'PositionCl', 'PositionHe', 'frame_reference', 'sj_nr_y', 'gender', 'age_y', 'edu', 'pol_autho_y', 'pol_scep_y', 'PositionImm', 'PositionCl', 'PositionHe', 'video_sharing', 'sj_nr', 'gender', 'age', 'edu', 'pol_autho', 'pol_scep', 'PositionImm', 'PositionCl', 'PositionHe']
=== NEGATIVE AFFECT MODEL ===
                            OLS Regression Results                            
Dep. Variable:             NEG_AFFECT   R-squared:                       0.100
Model:                            OLS   Adj. R-squared:                  0.090
Method:                 Least Squares   F-statistic:                     9.845
Date:                Sun, 19 Oct 2025   Prob (F-statistic):           9.54e-16
Time:                        20:50:32   Log-Likelihood:                -3877.8
N

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# Create the  plot 
fig = go.Figure()

# Define conditions and colors
conditions = ['Threat', 'Blame']
fear_color = '#56B4E9'   # light blue
anger_color = '#D55E00'  # dark pink / orange-red
error_color = '#222222'  # Black error bars
grid_color = '#D3D3D3'   # Light gray

# Add Fear data 
fig.add_trace(go.Scatter(
    name='Fear',
    x=conditions,
    y=[fear_coefs['threat']['coef'], fear_coefs['blame']['coef']],
    error_y=dict(
        type='data',
        array=[fear_coefs['threat']['se'], fear_coefs['blame']['se']],
        visible=True,
        thickness=3,
        width=6,
        color=error_color 
    ),
    mode='markers', 
    marker=dict(
        size=20, 
        color=fear_color,
        symbol='diamond',
        line=dict(width=3, color=error_color)
    ),
    hovertemplate='Fear - %{x}: %{y:.2f} ± %{error_y.array:.2f}<extra></extra>'
))

# Add Anger data 
fig.add_trace(go.Scatter(
    name='Anger',
    x=conditions,
    y=[anger_coefs['threat']['coef'], anger_coefs['blame']['coef']],
    error_y=dict(
        type='data',
        array=[anger_coefs['threat']['se'], anger_coefs['blame']['se']],
        visible=True,
        thickness=3,
        width=6,
        color=error_color 
    ),
    mode='markers', 
    marker=dict(
        size=20, 
        color=anger_color,
        symbol='diamond',
        line=dict(width=3, color=error_color)
    ),
    hovertemplate='Anger - %{x}: %{y:.2f} ± %{error_y.array:.2f}<extra></extra>'
))

# Calculate significance 
def get_significance_stars(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return 'n.s.'

# Add significance annotations 
y_offset = 1.5

# Fear significance 
fig.add_annotation(
    x='Threat', 
    y=fear_coefs['threat']['coef'] + fear_coefs['threat']['se'] + y_offset,
    text=get_significance_stars(fear_coefs['threat']['p_value']),
    showarrow=False, 
    font=dict(
        size=20 if get_significance_stars(fear_coefs['threat']['p_value']) != 'n.s.' else 14, 
        color='black', 
        family='Arial'  
    ),
    yshift=10
)

fig.add_annotation(
    x='Blame', 
    y=fear_coefs['blame']['coef'] + fear_coefs['blame']['se'] + y_offset,
    text=get_significance_stars(fear_coefs['blame']['p_value']),
    showarrow=False, 
    font=dict(
        size=20 if get_significance_stars(fear_coefs['blame']['p_value']) != 'n.s.' else 14,  
        color='black', 
        family='Arial'  =
    ),
    yshift=10
)

# Anger significance 
fig.add_annotation(
    x='Threat', 
    y=anger_coefs['threat']['coef'] + anger_coefs['threat']['se'] + y_offset,
    text=get_significance_stars(anger_coefs['threat']['p_value']),
    showarrow=False, 
    font=dict(
        size=20 if get_significance_stars(anger_coefs['threat']['p_value']) != 'n.s.' else 14, 
        color='black', 
        family='Arial'  
    ),
    yshift=10
)

fig.add_annotation(
    x='Blame', 
    y=anger_coefs['blame']['coef'] + anger_coefs['blame']['se'] + y_offset,
    text=get_significance_stars(anger_coefs['blame']['p_value']),
    showarrow=False, 
    font=dict(
        size=20 if get_significance_stars(anger_coefs['blame']['p_value']) != 'n.s.' else 14,  
        color='black', 
        family='Arial'  
    ),
    yshift=10
)

# Customize layout 
fig.update_layout(
    title=dict(

        x=0.05, 
        xanchor='left', 
        font=dict(size=22, family='Arial')  
    ),
    xaxis=dict(
        title="",
        tickmode='array',
        tickvals=conditions,
        ticktext=conditions,
        tickfont=dict(size=18, family='Arial'), 
        showgrid=False,
        range=[-0.3, 1.3]  
    ),
    yaxis=dict(
        title="Simplification frame impact<br>on negative arousal (a.u.)", 
        title_font=dict(size=20, family='Arial'),   
        tickfont=dict(size=16, family='Arial'), 
        gridcolor=grid_color,
        gridwidth=1,
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor='black'
    ),
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(size=16, family='Arial'), 
        bgcolor='rgba(255,255,255,0.8)'
    ),
    template="plotly_white",
    width=400,
    height=400,
    margin=dict(l=80, r=40, t=80, b=60)
)

fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig.show()


STUDY 2 COEFFICIENTS FOR EMOTIONAL AROUSAL
FEAR:
  Threat: β = 5.000, SE = 1.000, p = 0.4000
  Blame:  β = 4.000, SE = 1.000, p = 0.5000

ANGER:
  Threat: β = 17.000, SE = 1.500, p = 0.0003
  Blame:  β = 22.000, SE = 1.200, p = 0.0002


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# Create the plot 
fig = go.Figure()

# Define conditions a
conditions = ['Threat', 'Blame']
sharing_color = '#2C4A52'    # Dark teal (agreement/upper diamond)
importance_color = '#E6C87F' # Muted yellow-beige (importance/lower diamond)
error_color = '#222222'      # Black (slightly gray)
grid_color = '#D3D3D3'       # Light gray

# Extract coefficients 
threat_imp_coef = 2.7773
threat_imp_se = 2.660
blame_imp_coef = 5.2652
blame_imp_se = 2.505

threat_clap_coef = -1.2202
threat_clap_se = 0.827
blame_clap_coef = -2.1850
blame_clap_se = 0.779

# Adjust coefficients 
adjusted_threat_imp = threat_imp_coef - 0.5
adjusted_blame_imp = blame_imp_coef - 0.5
adjusted_threat_clap = threat_clap_coef - 0.5
adjusted_blame_clap = blame_clap_coef - 0.5

# Add Video Sharing data 
fig.add_trace(go.Scatter(
    name='Video Sharing',
    x=conditions,
    y=[adjusted_threat_clap, adjusted_blame_clap],
    error_y=dict(
        type='data',
        array=[threat_clap_se, blame_clap_se],
        visible=True,
        thickness=3,
        width=6,
        color=error_color  
    ),
    mode='markers',  
    marker=dict(
        size=20,  
        color=sharing_color,
        symbol='diamond',
        line=dict(width=3, color=error_color)
    ),
    hovertemplate='Video Sharing - %{x}: %{y:.2f} ± %{error_y.array:.2f}<extra></extra>'
))

# Add Issue Importance 
fig.add_trace(go.Scatter(
    name='Issue Importance',
    x=conditions,
    y=[adjusted_threat_imp, adjusted_blame_imp],
    error_y=dict(
        type='data',
        array=[threat_imp_se, blame_imp_se],
        visible=True,
        thickness=3,
        width=6,
        color=error_color 
    ),
    mode='markers',
    marker=dict(
        size=20, 
        color=importance_color,
        symbol='diamond',
        line=dict(width=3, color=error_color)
    ),
    hovertemplate='Issue Importance - %{x}: %{y:.2f} ± %{error_y.array:.2f}<extra></extra>'
))

# Calculate significance 
def get_significance_stars(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return 'n.s.'

# Add significance annotations 
y_offset = 1.0 

# Video Sharing 
fig.add_annotation(
    x='Threat', 
    y=adjusted_threat_clap + threat_clap_se + y_offset,
    text=get_significance_stars(0.140), 
    showarrow=False, 
    font=dict(
        size=20 if get_significance_stars(0.140) != 'n.s.' else 14, 
        color='black', 
        family='Arial'  
    ),
    yshift=10
)

fig.add_annotation(
    x='Blame', 
    y=adjusted_blame_clap + blame_clap_se + y_offset,
    text=get_significance_stars(0.005),  
    showarrow=False, 
    font=dict(
        size=20 if get_significance_stars(0.005) != 'n.s.' else 14,  
        color='black', 
        family='Arial'  
    ),
    yshift=10
)

# Issue Importance 
fig.add_annotation(
    x='Threat', 
    y=adjusted_threat_imp + threat_imp_se + y_offset,
    text=get_significance_stars(0.297),  
    showarrow=False, 
    font=dict(
        size=20 if get_significance_stars(0.297) != 'n.s.' else 14, 
        color='black', 
        family='Arial'  
    ),
    yshift=10
)

fig.add_annotation(
    x='Blame', 
    y=adjusted_blame_imp + blame_imp_se + y_offset,
    text=get_significance_stars(0.036),
    showarrow=False, 
    font=dict(
        size=20 if get_significance_stars(0.036) != 'n.s.' else 14,
        color='black', 
        family='Arial' 
    ),
    yshift=10
)

# Customize layout
fig.update_layout(
    title=dict(
        text="(b.) Study 2",  
        x=0.05,  
        xanchor='left', 
        font=dict(size=22, family='Arial')  
    ),
    xaxis=dict(
        title="",
        tickmode='array',
        tickvals=conditions,
        ticktext=conditions,
        tickfont=dict(size=18, family='Arial'),
        showgrid=False,
        range=[-0.3, 1.3]  
    ),
    yaxis=dict(
        title="Simplification frame impact<br>on political behaviour (a.u.)",  
        title_font=dict(size=20, family='Arial'),  
        tickfont=dict(size=16, family='Arial'),  
        gridcolor=grid_color,
        gridwidth=1,
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor='black'
    ),
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(size=16, family='Arial'),  
        bgcolor='rgba(255,255,255,0.8)'
    ),
    template="plotly_white",
    width=400,
    height=500,
    margin=dict(l=80, r=40, t=80, b=60)
)
fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig.show()



STUDY 2 COEFFICIENTS FOR POLITICAL BEHAVIOR
ORIGINAL COEFFICIENTS (before -0.5 adjustment):
ISSUE IMPORTANCE:
  Threat: β = 2.777, SE = 2.660, p = 0.297
  Blame:  β = 5.265, SE = 2.505, p = 0.036

VIDEO SHARING:
  Threat: β = -1.220, SE = 0.827, p = 0.140
  Blame:  β = -2.185, SE = 0.779, p = 0.005

Note: All y-values have been adjusted by -0.5 for visualization
